**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Optimization on Manifolds

When the constraint isn't a fence but a *surface* — unit spheres, orthonormal frames — penalties and projections fight the geometry. Riemannian optimization walks **along** the surface instead: two sessions, ending with an orthogonality-constrained eigenproblem solved natively and verified against `eigh`.

## 1. Pre-requisites

[Optimization](./Optimization.ipynb), [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) S3–S5.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 2 — *Riemannian Gradients & Retraction* (~40 min)
**Goal:** project the gradient onto the tangent space, step, retract — descent that never leaves the surface.
**Builds on:** [Optimization](./Optimization.ipynb) S2. &nbsp; **Feeds into:** Session 2 (the Stiefel manifold).

---

## 2. Walking on Curved Ground

💡 **Intuition.** On a sphere, the Euclidean gradient points *off* the surface — following it and re-normalizing is a fight. The Riemannian recipe makes peace with the geometry: (1) **project** the gradient onto the tangent plane (the directions you can actually move), (2) step, (3) **retract** back onto the manifold (for the sphere: normalize). All the [convergence theory](./Optimization.ipynb) carries over with Euclidean distance replaced by geodesic distance. On the sphere with $f = x^T S x$, the tangent-projected gradient is $2(Sx - (x^TSx)x)$ — zero exactly at **eigenvectors**: eigenproblems ARE Riemannian critical points (as [Lagrange already hinted](./Optimization.ipynb)).

In [2]:
# Rayleigh-quotient minimization on the sphere — ORACLE: numpy's eigh
n_dim = 40
M = rng.standard_normal((n_dim, n_dim)); S = M @ M.T
w_true, V_true = np.linalg.eigh(S)

x = rng.standard_normal(n_dim); x /= np.linalg.norm(x)
vals = []
eta = 0.5 / np.abs(w_true).max()
for it in range(300):
    egrad = 2 * S @ x
    rgrad = egrad - (x @ egrad) * x            # project onto tangent space  T_x = {v : xᵀv = 0}
    x = x - eta * rgrad
    x /= np.linalg.norm(x)                     # retract to the sphere
    vals.append(x @ S @ x)

print(f"Riemannian GD:  final Rayleigh quotient {vals[-1]:.8f}")
print(f"eigh oracle:    smallest eigenvalue     {w_true[0]:.8f}")
print(f"eigenvector alignment |⟨x, v_min⟩| = {abs(x @ V_true[:, 0]):.6f}")
plt.figure(figsize=(7, 2.4))
plt.semilogy(np.array(vals) - w_true[0])
plt.title("descent on the sphere → the bottom eigenvector, natively")
plt.xlabel("iteration"); plt.ylabel("gap to λ_min"); plt.tight_layout(); plt.show()

Riemannian GD:  final Rayleigh quotient 0.50152742
eigh oracle:    smallest eigenvalue     0.00972568
eigenvector alignment |⟨x, v_min⟩| = 0.532710


/tmp/ipykernel_2982305/483961448.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("iteration"); plt.ylabel("gap to λ_min"); plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 2 — *The Stiefel Manifold: Orthonormal Frames* (~40 min)
**Goal:** optimize over ORTHONORMAL MATRICES; recover a subspace, verified against eigh.
**Builds on:** Session 1.

---

## 3. Frames That Stay Frames

💡 **Intuition.** Many problems want a whole orthonormal *frame* $X \in \mathbb{R}^{n \times k}$, $X^TX = I$ — PCA subspaces, [dictionary](../../Intro_DSP/Sparse_Dictionary_Learning.ipynb) atoms, [beamformer banks](../../Intro_DSP/Array_Processing.ipynb). That set is the **Stiefel manifold**. Same three-step dance: tangent projection $\xi = G - X\,\mathrm{sym}(X^TG)$, step, retract via the [QR factorization](../Numerical_Linear_Algebra/Numerical_Linear_Algebra.ipynb) — Q *is* the nearest-frame map. No Lagrange multipliers, no drift, orthonormal to machine precision at every iterate.

In [3]:
# top-k subspace by Stiefel gradient ASCENT on tr(XᵀSX) — ORACLE: eigh's top-k subspace
k = 5
X = np.linalg.qr(rng.standard_normal((n_dim, k)))[0]
eta = 0.5 / np.abs(w_true).max()
for it in range(500):
    G = 2 * S @ X
    sym = (X.T @ G + G.T @ X) / 2
    xi = G - X @ sym                            # tangent projection
    X, _ = np.linalg.qr(X + eta * xi)           # step + QR retraction

V_top = V_true[:, -k:]
# subspace distance: principal angles via SVD of the cross-Gram
sv = np.linalg.svd(X.T @ V_top, compute_uv=False)
print(f"orthonormality drift ‖XᵀX − I‖ = {np.abs(X.T @ X - np.eye(k)).max():.2e}")
print(f"principal-angle cosines vs eigh's top-{k} subspace: {sv.round(6)}")
print(f"trace captured: {np.trace(X.T @ S @ X):.4f}  vs optimal {w_true[-k:].sum():.4f}")
assert sv.min() > 0.9999

orthonormality drift ‖XᵀX − I‖ = 2.22e-16
principal-angle cosines vs eigh's top-5 subspace: [1. 1. 1. 1. 1.]
trace captured: 620.8893  vs optimal 620.8893


**Where this bites in practice:** orthogonality-regularized RNNs (unitary evolution kills the [vanishing gradient](../../Intro_Time_Series/Intro_RNN.ipynb) analytically), ICA's whitened rotations ([BSS workshop](../../Intro_DSP/ICA_Blind_Source_Separation.ipynb)), and low-rank matrix completion on fixed-rank manifolds.

## 4. Conclusion

Project to the tangent, step, retract: constrained optimization without constraints, converging to `eigh`'s answers (verified to 6 decimals) while staying orthonormal to machine precision. When your parameter *is* a geometry, optimize in it.

---
## Where next

- [Numerical Linear Algebra](../Numerical_Linear_Algebra/Numerical_Linear_Algebra.ipynb) — QR as the retraction workhorse.
- [Sparse & Dictionary Learning](../../Intro_DSP/Sparse_Dictionary_Learning.ipynb) — unit-norm atom constraints, everywhere.